# Classroom Environment Setup

Module: Setup and Python Ecosystem

## Lesson summary

This classroom setup notebook is the short pre-session smoke test. It assumes
the first-clone setup in `0.3.initial_repository_setup` and the full validation
in `0.2.environment_validation_lab` have already passed. Its job is to answer
one practical question before a live class: is this kernel ready to run the
next data notebook without downloading market data or exposing credentials
{cite}`pyenv2025,uv2025,kluyver2016jupyter`.

## Learning objectives

By the end of this note, readers should be able to:

- confirm that the classroom environment has been synchronized from `uv.lock`;
- verify data, finance, visualization, widget, and storage libraries such as `pandas`, `yfinance`, `requests`, `plotly`, `ipywidgets`, and `pyarrow`;
- confirm that snapshot-backed lessons can find the local snapshot manifest;
- create a concise environment record that can accompany reproducibility questions.

## Prerequisites

The fresh-clone setup and full environment validation must already pass. Run
`uv sync` after any lockfile change and launch this notebook from the project
environment; no live provider credential is needed for the smoke test.

## Lesson flow

1. Open the repository from the project environment.
2. Run the compact readiness report.
3. Record package versions with `watermark`.
4. Move to data notebooks only after the check passes.

## Setup

Open the notebook from JupyterLab after running `uv sync`; the runtime cells should execute without downloading market data or printing credentials.

## Terminal setup with pyenv and uv

Run these commands from the repository root before opening JupyterLab. The project dependencies are declared in `pyproject.toml` and installed by `uv sync`.

```bash
pyenv install 3.12.12
pyenv local 3.12.12
uv sync
uv run jupyter lab
```

## Session readiness check

Use this check before running data notebooks that depend on external providers, interactive widgets, or columnar storage libraries. It is a smoke test, not a replacement for the full environment validation lab.

In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
for candidate in (project_root, *project_root.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "uv.lock").exists():
        project_root = candidate
        break

classroom_packages = {
    "pandas": "pandas",
    "yfinance": "yfinance",
    "requests": "requests",
    "plotly": "plotly",
    "ipywidgets": "ipywidgets",
    "pyarrow": "pyarrow",
}

package_status = {}
for import_name, distribution_name in classroom_packages.items():
    try:
        package_status[import_name] = version(distribution_name)
    except PackageNotFoundError:
        package_status[import_name] = "missing"

readiness_records = [
    {"check": "python_3_12_or_newer", "ready": sys.version_info >= (3, 12), "detail": sys.version.split()[0]},
    {"check": "project_root", "ready": (project_root / "pyproject.toml").exists(), "detail": str(project_root)},
    {"check": "uv_lock", "ready": (project_root / "uv.lock").exists(), "detail": "uv.lock"},
    {
        "check": "published_notebooks",
        "ready": (project_root / "notebooks" / "course").exists(),
        "detail": "notebooks/course",
    },
    {"check": "snapshot_manifest", "ready": (project_root / "data" / "snapshots" / "metadata.json").exists(), "detail": "data/snapshots/metadata.json"},
]
readiness_records.extend(
    {"check": f"package:{name}", "ready": package_version != "missing", "detail": package_version}
    for name, package_version in package_status.items()
)

session_readiness = pd.DataFrame(readiness_records)
assert session_readiness["ready"].all(), session_readiness
session_readiness

**Output interpretation.**

Every `ready` value should be `True`. This table is deliberately compact: it confirms that the kernel, repository root, lockfile, notebook folder, snapshot manifest, and classroom packages are available before class starts.

## Package version record

The watermark output gives a compact reproducibility stamp for the active classroom session.

In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w -p pandas,numpy,yfinance,requests,aleatory,pyarrow,plotly,ipywidgets

**Output interpretation.**

The watermark stamp records package versions and the last update time for the notebook run. If a classroom issue appears later, this output helps separate content problems from local environment drift.

## Handoff

A fully `True` readiness table is the final Module 0 gate. Preserve the table
with classroom issue reports, then continue to Module 1's market foundations
and source inventory. Return to the full validation lab if the problem
involves helpers, snapshots, or credentials beyond this smoke test.